# Práctica 1 — Parte E: acceso a GPU
Cómputo Paralelo · 6BM1 · 27/1. Ejecutar en Colab (Entorno de ejecución → T4 GPU) o Kaggle (Accelerator → GPU).
Guardar este notebook **con salidas** en .

In [ ]:
# Celda 1 — ¿qué GPU me tocó? (pega una captura de esta salida en el reporte, con fecha)
!nvidia-smi
!nvcc --version | tail -1
import time
import numpy as np
import cupy as cp
print(cp.cuda.runtime.getDeviceProperties(0)["name"].decode(), "| CuPy", cp.__version__)

In [ ]:
# Celda 2 — medición correcta en GPU: la GPU trabaja de forma asíncrona,
# así que hay que sincronizar antes de leer el reloj
def medir(fn, reps=5, sync=None):
    fn()                                   # calentamiento (compila kernels, reserva memoria)
    if sync: sync()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn()
        if sync: sync()
        ts.append(time.perf_counter() - t0)
    return np.mean(ts), np.std(ts)

sync = cp.cuda.Device().synchronize
for n in [256, 1024, 4096]:
    A = np.random.rand(n, n).astype(np.float32); B = np.random.rand(n, n).astype(np.float32)
    Ag, Bg = cp.asarray(A), cp.asarray(B)  # copia a la GPU (fuera de la región medida)
    t_cpu, s_cpu = medir(lambda: A @ B)
    t_gpu, s_gpu = medir(lambda: Ag @ Bg, sync=sync)
    t_mal, _     = medir(lambda: Ag @ Bg)          # sin sincronizar: mal medido a propósito
    print(f"n={n:5d}  CPU {t_cpu:.4f}±{s_cpu:.4f} s   GPU {t_gpu:.5f}±{s_gpu:.5f} s   "
          f"speedup {t_cpu / t_gpu:7.1f}x   |  GPU sin sync {t_mal:.6f} s (incorrecto)")